In [ ]:
import pandas as pd
import numpy as np 

In [ ]:
from sklearn.ensemble import RandomForestRegressor
def get_rf_imputations_sep(journal, seeds, outcome = "Times cited (36mo)", pred_type = "basic"):
    filename = f"open_access_results/{journal}_qualities.csv"
    results = pd.read_csv(filename)
    results[['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']] = results['response'].str.split(',', expand=True, n =10).apply(lambda lst: [x.strip() for x in lst])
    journal_df = pd.read_csv(f"open_access_journal_data/{journal}.csv")
    for col in ['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']:

        transform_df = pd.concat([
            results.loc[results[col] == 'Paper 1', 'Article Title.1'],
            results.loc[results[col] == 'Paper 2', 'Article Title.2']
        ])

        rate_df = transform_df.value_counts().reset_index()
        rate_df.columns = ['Article Title', f'{col}_score']

        journal_df = pd.merge(journal_df, rate_df, on = 'Article Title', how = 'left').fillna(0)
    filename = f"open_access_results/{journal}_basic.csv"
    results = pd.read_csv(filename)
    transform_df = pd.concat([
        results.loc[results['response'] == 'Paper 1', 'Article Title.1'],
        results.loc[results['response'] == 'Paper 2', 'Article Title.2']
        ])

    rate_df = transform_df.value_counts().reset_index()
    rate_df.columns = ['Article Title', 'risk_score']
    rate_df.sort_values('Article Title').head()
    rate_df = rate_df[['Article Title', 'risk_score']]

    journal_df = pd.merge(journal_df, rate_df, on = "Article Title", how = 'left').fillna(0)

    journal_df['log_authors'] = np.log(journal_df['Authors'])
    journal_df['log_pages'] = np.log(journal_df['Page length'])
    journal_df['review'] = np.where(journal_df['Article type'] == "Review", 1, 0) 
    journal_df['log_outcome'] = np.log(journal_df[outcome] + 1)

    if pred_type == "basic":
        X = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'Open Access']]
    elif pred_type == "score_only":
        X = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'Open Access', 'risk_score']]
    elif pred_type == "qualities_only":
        X = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'topic_novelty_score',
       'topic_popularity_score', 'title_catchiness_score',
       'generalizability_score', 'writing_quality_score',
       'impact_of_results_score', 'subfield_popularity_score',
       'technicality_score', 'meaningful_contributions_score', 'applicability_score', 'Open Access']]
    elif pred_type == "all":
        X = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'topic_novelty_score',
       'topic_popularity_score', 'title_catchiness_score',
       'generalizability_score', 'writing_quality_score',
       'impact_of_results_score', 'subfield_popularity_score',
       'technicality_score', 'meaningful_contributions_score', 'applicability_score', 'risk_score', 'Open Access']]
        
    y = journal_df['log_outcome']

    # drop each observation, refit the random forest, and then get control and treatment inmputations 
    for i in journal_df.index:

        seed = int(seeds[i])

        X_withouti = X.drop(index = i)
        y_withouti = np.delete(y, i)

        model_withouti = RandomForestRegressor(
        n_estimators=100,
        oob_score=True,
        bootstrap=True,   # must be True for OOB
        random_state=seed
        )

        model_withouti.fit(X_withouti, y_withouti)

        observation_it = X.iloc[[i]]
        observation_it.loc[:,'Open Access'] = 1 

        observation_ic = X.iloc[[i]]
        observation_ic.loc[:,'Open Access'] = 0

        y_it = model_withouti.predict(observation_it)[0]
        y_ic = model_withouti.predict(observation_ic)[0]
    
        journal_df.loc[i, 'y_ic'] = y_ic
        journal_df.loc[i, 'y_it'] = y_it

    # save imputed data frames 
    journal_df.to_csv(f"open_access_results/{journal}_{pred_type}_imputations.csv")

    return journal_df

def get_variance(journal_df, p):
    journal_df['m'] = p * journal_df['y_it'] + (1- p) * journal_df['y_ic']

    nc = len(journal_df) - sum(journal_df['Open Access'])
    nt = sum(journal_df['Open Access'])
    N = len(journal_df)

    ec2 = 1/nc * sum((1 - journal_df['Open Access']) * (journal_df['y_ic'] - journal_df['log_outcome'])**2)
    et2 = 1/nt * sum(journal_df['Open Access'] * (journal_df['y_it'] - journal_df['log_outcome'])**2)

    var_est = 1/N * (p/(1-p) * ec2 + (1-p)/p * et2 + 2 * (ec2 * et2)**0.5)
    return var_est


rows = []
master_seed = 25
rng = np.random.default_rng(master_seed)
seed_list = iter(rng.integers(low=0, high=2**32 - 1, size =20))
ps = {'science': 0.12, 'neuro': 0.14, 'genetics': 0.49, 'faseb': 0.49, 'physio': 0.13}
sizes = {'science': 393, 'neuro': 278, 'genetics': 211, 'faseb': 165, 'physio': 201} 
for journal in ['science', 'neuro', 'genetics', 'faseb', 'physio']:
    print(journal)
    row = []
    for pred_type in ['basic', 'score_only', 'qualities_only', 'all']:
        print(pred_type)
        rng = np.random.default_rng(int(next(seed_list)))
        seeds = rng.integers(low=0, high=2**32 - 1, size =sizes[journal])
        journal_df = get_rf_imputations_sep(journal = journal, pred_type = pred_type, seeds= seeds)
        var_est = get_variance(journal_df, ps[journal])
        row.append(var_est)
    rows.append(row) 

results = pd.DataFrame(rows, index = ['Science', 'Neurophysiology', 'Genetics', 'FASEB','Applied Physiology'], columns =['Base Covariates', 'Base + Rating Score', 'Base + 10 Qualities', 'Base + Both'] )
print(np.sqrt(results))